# Lesson 18: Multi-Model Edge Benchmarking — TTFT, Throughput, Perplexity & SLM vs. Cloud LLM Trade-offs

In this companion notebook, you will:
1. Benchmark open-source local SLMs (`Qwen/Qwen2.5-0.5B-Instruct` vs. `HuggingFaceTB/SmolLM2-360M-Instruct`) on **Time-To-First-Token (TTFT)**, **Tokens Per Second (TPS)**, and **p95 latency**.
2. Optionally compare against **Google AI Studio Free Tier** (`gemini-2.5-flash`) with deterministic fallback.
3. Export a structured Markdown benchmark report.

In [ ]:
!pip install -q -U transformers torch google-genai

## 1. Benchmark Two Compact Open-Source SLMs
Measure cold-start warmup and steady-state decode throughput across repeated rounds.

In [ ]:
import time
import statistics
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

models_to_bench = [
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct",
]
prompt = "Explain why edge AI preserves privacy in one sentence."

results = []
for mid in models_to_bench:
    tok = AutoTokenizer.from_pretrained(mid)
    mdl = AutoModelForCausalLM.from_pretrained(mid)
    inputs = tok(prompt, return_tensors="pt")
    latencies = []
    gen_tokens = 32
    for _ in range(2):
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = mdl.generate(**inputs, max_new_tokens=gen_tokens, do_sample=False)
        latencies.append(time.perf_counter() - t0)
    avg_s = statistics.mean(latencies)
    results.append((mid, avg_s, gen_tokens / avg_s))

print("| Model | Avg Latency (s) | Throughput (tok/s) |")
print("| :--- | :---: | :---: |")
for mid, avg_s, tps in results:
    print(f"| `{mid}` | {avg_s:.2f} | {tps:.1f} |")